# Cross-model necessity scan (class × size) — API / Bedrock

Runs the `tab:disc` cross-model result over a **model matrix**, logging **every exact command
with a UTC timestamp** for traceability, plus provenance (git commit, region, model ids). No GPU —
closed models over AWS Bedrock. ~a few min/model at 30 rpm.


## 1 · Config + clone (edit `MODELS`, then Run All)
Model ids below are **verified present** on the account that produced this list — re-check with
`aws bedrock list-inference-profiles` if you switch account/region.

In [ ]:
import os, re, json, subprocess, datetime

REPO_URL   = 'https://github.com/wrgr/socratic-scenarios.git'
BRANCH     = 'main'
AWS_REGION = os.environ.get('AWS_REGION', 'us-east-1')

# (label, bedrock_model_id) — labels 'class-size' so the table sorts by tier. All ids verified present.
MODELS = [
    # Anthropic Claude, 4.5 generation:  haiku < sonnet < opus
    ('claude-small',  'us.anthropic.claude-haiku-4-5-20251001-v1:0'),
    ('claude-medium', 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'),
    ('claude-large',  'us.anthropic.claude-opus-4-5-20251101-v1:0'),
    # Meta Llama:  8B < 70B < 90B   (no 405B offered on this account)
    ('llama-small',   'us.meta.llama3-1-8b-instruct-v1:0'),
    ('llama-medium',  'us.meta.llama3-3-70b-instruct-v1:0'),
    ('llama-large',   'us.meta.llama3-2-90b-instruct-v1:0'),
    # Amazon Nova:  micro < lite < pro
    ('nova-small',    'us.amazon.nova-micro-v1:0'),
    ('nova-medium',   'us.amazon.nova-lite-v1:0'),
    ('nova-large',    'us.amazon.nova-pro-v1:0'),
    # --- optional extras available on this account (uncomment to include) ---
    # ('deepseek-r1',   'us.deepseek.r1-v1:0'),
    # ('writer-x5',     'us.writer.palmyra-x5-v1:0'),
    # ('llama4-mav',    'us.meta.llama4-maverick-17b-instruct-v1:0'),
    # ('claude-opus5',  'us.anthropic.claude-opus-5'),      # newest gen; note: no -v1:0 suffix
    # ('claude-sonnet5','us.anthropic.claude-sonnet-5'),
]
PROBES_SETS = ['hazard', 'all']

if not os.path.isdir('socratic-scenarios'):
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL], check=True)
subprocess.run(['git','fetch','--depth','1','origin',BRANCH], cwd='socratic-scenarios', check=True)
subprocess.run(['git','reset','--hard','FETCH_HEAD'], cwd='socratic-scenarios', check=True)
REPO = os.path.abspath('socratic-scenarios')
print('repo:', REPO, '| region:', AWS_REGION, '| models:', len(MODELS))


## 2 · Node deps + AWS credentials
Credentials come from your **standard AWS chain** (env vars already set, `~/.aws`, or an
instance/role). No paste-a-key cell by design — set creds your usual way before running.

In [ ]:
subprocess.run(['npm','install','--no-audit','--no-fund','--loglevel=error'], cwd=REPO, check=True)
who = subprocess.run(['aws','sts','get-caller-identity'], capture_output=True, text=True)
if who.returncode == 0:
    print('AWS identity OK:', json.loads(who.stdout).get('Arn','?'))
else:
    have = [k for k in ('AWS_ACCESS_KEY_ID','AWS_PROFILE','AWS_ROLE_ARN','AWS_WEB_IDENTITY_TOKEN_FILE') if os.environ.get(k)]
    print('aws cli check unavailable; env-chain markers present:', have or 'NONE — set creds before the scan')


## 3 · Provenance (saved with the results)

In [ ]:
run_utc = datetime.datetime.now(datetime.timezone.utc)
commit  = subprocess.run(['git','rev-parse','HEAD'], cwd=REPO, capture_output=True, text=True).stdout.strip()
PROV = {
    'run_utc':   run_utc.isoformat(),
    'run_local': datetime.datetime.now().astimezone().isoformat(),
    'git_commit': commit, 'branch': BRANCH, 'aws_region': AWS_REGION,
    'models': dict(MODELS), 'probe_sets': PROBES_SETS,
    'instrument': 'colreg:leakage (necessity=regret-delta; redundant/unusable=regret-with)',
}
print(json.dumps(PROV, indent=2))


## 4 · Run the scan  (each exact command is logged with a UTC timestamp)

In [ ]:
DASH = r'[-−]'   # ASCII hyphen or unicode minus

def parse_hazard(out):
    r = {}
    m = re.search(r'regret-delta\s+[\d.]+\s*\(without\)\s*'+DASH+r'\s*([\d.]+)\s*\(with\)\s*=\s*(-?[\d.]+)', out)
    if m: r['necessity'] = float(m.group(2)); r['regret_with'] = float(m.group(1))
    v = re.search(r'VERDICT:\s*([A-Z-]+)(?:\s*\((\w+):\s*regret-with\s*([\d.]+)\))?', out)
    if v:
        r['verdict'] = v.group(1); r['leak_mode'] = v.group(2) or ''
        if v.group(3): r['regret_with'] = float(v.group(3))
    return r

def parse_all(out):
    r = {}
    s = re.search(r'(\d+)\s*rules?\s*·\s*(\d+)\s*relied-on.*?·\s*(\d+)\s*redundant.*?·\s*(\d+)\s*unusable.*?·\s*(\d+)\s*inconclusive', out)
    if s: r.update(rules=int(s.group(1)), relied=int(s.group(2)), redundant=int(s.group(3)), unusable=int(s.group(4)), inconclusive=int(s.group(5)))
    suff = re.search(r'(FALSE SUFFICIENCY|CONTRIBUTING|UNUSABLE|PARTIAL)', out)
    if suff: r['sufficiency'] = suff.group(1)
    return r

outdir = os.path.join(REPO, 'results', 'model-scan'); os.makedirs(outdir, exist_ok=True)
run_log = []   # traceability: exact command + UTC per call

def run_one(label, model, probes):
    ts  = datetime.datetime.now(datetime.timezone.utc).isoformat()
    cmd = f"AWS_REGION={AWS_REGION} BEDROCK_MODEL={model} PROBES={probes} npm run --silent colreg:leakage"
    print(f"RUN {ts}  [{label:13}] {cmd}", flush=True)
    run_log.append({'utc': ts, 'label': label, 'model': model, 'probes': probes, 'cmd': cmd})
    env = dict(os.environ, AWS_REGION=AWS_REGION, BEDROCK_MODEL=model, PROBES=probes)
    return subprocess.run(['npm','run','--silent','colreg:leakage'], cwd=REPO, env=env, capture_output=True, text=True)

rows = []
for label, model in MODELS:
    row = {'label': label, 'model': model}
    for probes in PROBES_SETS:
        p = run_one(label, model, probes)
        out = (p.stdout or '') + '\n' + (p.stderr or '')
        open(os.path.join(outdir, f'{label}__{probes}.txt'), 'w').write(out)
        if 'VERDICT' not in out and p.returncode != 0:
            row[f'{probes}_error'] = (p.stderr or p.stdout or 'failed')[-200:]
        elif probes == 'hazard': row.update(parse_hazard(out))
        else: row.update(parse_all(out))
    rows.append(row)
    err = '  ERR' if any(k.endswith('_error') for k in row) else ''
    print(f"  -> [{label:13}] hazard={row.get('verdict','?')}/{row.get('leak_mode','')} "
          f"necessity={row.get('necessity','?')} regret-with={row.get('regret_with','?')} | "
          f"std relied={row.get('relied','?')} redundant={row.get('redundant','?')} suff={row.get('sufficiency','?')}{err}\n")


## 5 · Summary + paste-back (provenance + full command log)

In [ ]:
def to_md(rows, cols):
    cols = [c for c in cols if any(c in r for r in rows)]
    head = '| ' + ' | '.join(cols) + ' |'
    sep  = '| ' + ' | '.join('---' for _ in cols) + ' |'
    body = '\n'.join('| ' + ' | '.join(str(r.get(c,'')) for c in cols) + ' |' for r in rows)
    return '\n'.join([head, sep, body])

stamp = run_utc.strftime('%Y%m%dT%H%M%SZ')
payload = {'provenance': PROV, 'command_log': run_log, 'rows': rows}
json.dump(payload, open(os.path.join(outdir, f'scan_{stamp}.json'), 'w'), indent=2)

cols = ['label','verdict','leak_mode','necessity','regret_with','relied','redundant','unusable','inconclusive','sufficiency']
print('==================== PASTE THIS BACK ====================')
print(f"cross-model necessity scan | run_utc={PROV['run_utc']} | local={PROV['run_local']}")
print(f"commit={PROV['git_commit'][:9]} branch={BRANCH} region={AWS_REGION} | {len(run_log)} commands logged")
print()
print(to_md(rows, cols))
print('\n<details><summary>provenance + command log + rows (machine-readable)</summary>\n')
print('```json'); print(json.dumps(payload, indent=2)); print('```\n</details>')
print(f"\nsaved: results/model-scan/scan_{stamp}.json  (+ per-model raw .txt)")


## Done
Paste the **PASTE THIS BACK** block into the chat — it carries the run timestamp, git commit, and
the full command log, so the `tab:disc` grid is fully traceable.